In [3]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np
import re
import os
import sys

def load_and_process_data(filepath):
    """
    Loads and processes simulation data from either a '_trace_matched_timing.csv' 
    or a '_trace.csv' file.

    For '_trace.csv' files, it parses issue and callback events, merging them
    to create a dataframe with issue_tick and callback_tick for each operation.

    Args:
        filepath (str): The path to the CSV file.

    Returns:
        pd.DataFrame: A DataFrame containing the processed data with columns 
                      including 'sys_id', 'node_name', 'issue_tick', 
                      'callback_tick', and 'elapsed_time'.
    """
    if '_trace.csv' in filepath:
        # Custom parser for the log-like trace format
        data = []
        header = None
        with open(filepath, 'r') as f:
            for line in f:
                # Use a more general regex to find the start of the CSV data
                csv_part_match = re.search(r',(.+)', line)
                if not csv_part_match:
                    continue
                
                csv_part = csv_part_match.group(1).strip()
                # The first part of the CSV is the action or the first header column
                parts = [line.split(',')[1].strip()] + csv_part.split(',')
                
                # The header line has a different structure
                if 'action' in parts[0]:
                    header = [h.strip() for h in parts]
                    # Fix header for callback which has 'callback_tick'
                    if 'issue_tick' in header:
                        header.append('callback_tick')
                    continue

                # If header is not set yet, skip data lines
                if header is None:
                    continue

                # Process data lines (issue/callback)
                if len(parts) < 3: # Basic check for valid line
                    continue

                action = parts[0].strip()
                if action not in ['issue', 'callback']:
                    continue

                row_data = {h: v for h, v in zip(header, parts)}
                
                # Callback lines have the tick value at the end
                if action == 'callback':
                    row_data['callback_tick'] = parts[-1].strip()
                
                data.append(row_data)

        df = pd.DataFrame(data)

        # Convert relevant columns to numeric, coercing errors
        for col in ['sys_id', 'node_id', 'issue_tick', 'callback_tick']:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        # Separate issue and callback events
        issues = df[df['action'] == 'issue'].copy()
        callbacks = df[df['action'] == 'callback'].copy()

        # Merge issues and callbacks on sys_id and node_id
        # We keep only the necessary columns to avoid conflicts
        merged = pd.merge(
            issues[['sys_id', 'node_id', 'node_name', 'issue_tick', 'node_type']],
            callbacks[['sys_id', 'node_id','callback_tick']],
            on=['sys_id', 'node_id'],
            how='left'
        )
        
        # Calculate elapsed time
        merged['elapsed_time'] = merged['callback_tick'] - merged['issue_tick']
        return merged
    
    elif '_trace_matched_timing.csv' in filepath:
        # Standard loading for pre-matched timing files
        return pd.read_csv(filepath)
    else:
        raise ValueError(f"Unsupported file type: {filepath}. Must be a '..._trace.csv' or '..._trace_matched_timing.csv' file.")


# --- 1. Load and Prepare Data ---

# Define file paths for the two simulation runs
# NOTE: You can now switch these to '_trace.csv' files if needed.
g2_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/GPT_3_1300M_grouped/GPT_3_1300M_multiple_1_2_4_2_1.seq_2048.batch_1024/run_20251216_100350_974ms/g2/GPT_3_1300M_multiple_1_2_4_2_1.seq_2048.batch_1024_trace_matched_timing.csv'
ns3_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/GPT_3_1300M_grouped/GPT_3_1300M_multiple_1_2_4_2_1.seq_2048.batch_1024/run_20251216_100353_558ms/ns3/GPT_3_1300M_multiple_1_2_4_2_1.seq_2048.batch_1024_trace_matched_timing.csv'
analytical_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/GPT_3_1300M_grouped/GPT_3_1300M_multiple_1_2_4_2_1.seq_2048.batch_1024/run_20251216_100348_086ms/analytical_unaware/GPT_3_1300M_multiple_1_2_4_2_1.seq_2048.batch_1024_trace_matched_timing.csv'

# g2_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/GPT_3_1300M_grouped/GPT_3_1300M_multiple_1_8_1_2_1.seq_2048.batch_1024/run_20251216_100350_755ms/g2/GPT_3_1300M_multiple_1_8_1_2_1.seq_2048.batch_1024_trace_matched_timing.csv'
# ns3_filepath =  '/app/astra-sim/upc/output/comparison_run/FoldedClos/GPT_3_1300M_grouped/GPT_3_1300M_multiple_1_8_1_2_1.seq_2048.batch_1024/run_20251216_100354_545ms/ns3/GPT_3_1300M_multiple_1_8_1_2_1.seq_2048.batch_1024_trace.csv'
# analytical_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/GPT_3_1300M_grouped/GPT_3_1300M_multiple_1_8_1_2_1.seq_2048.batch_1024/run_20251216_100348_084ms/analytical_unaware/GPT_3_1300M_multiple_1_8_1_2_1.seq_2048.batch_1024_trace_matched_timing.csv'


g2_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/GPT_3_1300M_grouped/GPT_3_1300M_multiple_2_4_1_2_0.seq_2048.batch_1024/run_20251216_100350_765ms/g2/GPT_3_1300M_multiple_2_4_1_2_0.seq_2048.batch_1024_trace_matched_timing.csv'
ns3_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/GPT_3_1300M_grouped/GPT_3_1300M_multiple_2_4_1_2_0.seq_2048.batch_1024/run_20251216_100354_406ms/ns3/GPT_3_1300M_multiple_2_4_1_2_0.seq_2048.batch_1024_trace_matched_timing.csv'
analytical_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/GPT_3_1300M_grouped/GPT_3_1300M_multiple_2_4_1_2_0.seq_2048.batch_1024/run_20251216_100348_090ms/analytical_unaware/GPT_3_1300M_multiple_2_4_1_2_0.seq_2048.batch_1024_trace_matched_timing.csv'

analytical_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Small_grouped_ecmp/T5_Small_multiple_1_8_2_1_0.seq_2048.batch_1024/run_20260106_105714_824ms/analytical_unaware/T5_Small_multiple_1_8_2_1_0.seq_2048.batch_1024_trace_matched_timing.csv'
g2_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Small_grouped_ecmp/T5_Small_multiple_1_8_2_1_0.seq_2048.batch_1024/run_20260106_105723_072ms/g2/T5_Small_multiple_1_8_2_1_0.seq_2048.batch_1024_trace_matched_timing.csv'
ns3_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Small_grouped_ecmp/T5_Small_multiple_1_8_2_1_0.seq_2048.batch_1024/run_20260106_105731_600ms/ns3/T5_Small_multiple_1_8_2_1_0.seq_2048.batch_1024_trace_matched_timing.csv'

analytical_filepath= '/app/astra-sim/upc/output/comparison_run/FoldedClos128/T5_Small_grouped_128/T5_Small_multiple_1_16_8_1_0.seq_2048.batch_1024/run_20260111_021242_322ms/analytical_unaware/T5_Small_multiple_1_16_8_1_0.seq_2048.batch_1024_trace_matched_timing.csv'
g2_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos128/T5_Small_grouped_128/T5_Small_multiple_1_16_8_1_0.seq_2048.batch_1024/run_20260111_021246_515ms/g2/T5_Small_multiple_1_16_8_1_0.seq_2048.batch_1024_trace_matched_timing.csv'
ns3_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos128/T5_Small_grouped_128/T5_Small_multiple_1_16_8_1_0.seq_2048.batch_1024/run_20260111_021349_326ms/ns3/T5_Small_multiple_1_16_8_1_0.seq_2048.batch_1024_trace_matched_timing.csv'

analytical_filepath='/app/astra-sim/upc/output/comparison_run/Dragonfly/T5_Small_grouped_ecmp/T5_Small_multiple_1_16_1_1_0.seq_2048.batch_1024/run_20260116_165908_222ms/analytical_unaware/T5_Small_multiple_1_16_1_1_0.seq_2048.batch_1024_trace_matched_timing.csv'
g2_filepath = '/app/astra-sim/upc/output/comparison_run/Dragonfly/T5_Small_grouped_ecmp/T5_Small_multiple_1_16_1_1_0.seq_2048.batch_1024/run_20260116_165909_386ms/g2/T5_Small_multiple_1_16_1_1_0.seq_2048.batch_1024_trace_matched_timing.csv'
ns3_filepath = '/app/astra-sim/upc/output/comparison_run/Dragonfly/T5_Small_grouped_ecmp/T5_Small_multiple_1_16_1_1_0.seq_2048.batch_1024/run_20260116_165911_075ms/ns3/T5_Small_multiple_1_16_1_1_0.seq_2048.batch_1024_trace_matched_timing.csv'

# Load the CSV files into pandas DataFrames using the new function
try:
    g2_df = load_and_process_data(g2_filepath)
    ns3_df = load_and_process_data(ns3_filepath)
    analytical_df = load_and_process_data(analytical_filepath)
    print("✅ Successfully loaded and processed g2, ns3, and analytical data files.")
except (FileNotFoundError, ValueError) as e:
    print(f"❌ Error loading or processing files: {e}. Please ensure the file paths are correct.")
    # Stop execution if files are not found/supported
    raise

# Filter for communication nodes only (node_type can be string or int)
g2_df['node_type'] = pd.to_numeric(g2_df['node_type'], errors='coerce')
ns3_df['node_type'] = pd.to_numeric(ns3_df['node_type'], errors='coerce')
analytical_df['node_type'] = pd.to_numeric(analytical_df['node_type'], errors='coerce')
comm_node_types = [0, 1,2,3,4,5, 6, 7]
g2_comm = g2_df[g2_df['node_type'].isin(comm_node_types)].copy()
ns3_comm = ns3_df[ns3_df['node_type'].isin(comm_node_types)].copy()
analytical_comm = analytical_df[analytical_df['node_type'].isin(comm_node_types)].copy()
print(f"Filtered for communication nodes. Found {len(g2_comm)} comm nodes in g2, {len(ns3_comm)} in ns3, and {len(analytical_comm)} in analytical.")

# --- 2. Merge DataFrames for Comparison ---

# Select and rename columns for a clean merge
g2_subset = g2_comm[['sys_id', 'node_id', 'node_name', 'issue_tick', 'callback_tick', 'elapsed_time']]
ns3_subset = ns3_comm[['sys_id', 'node_id', 'node_name', 'issue_tick', 'callback_tick', 'elapsed_time']]
analytical_subset = analytical_comm[['sys_id', 'node_id', 'node_name', 'issue_tick', 'callback_tick', 'elapsed_time']]

# Rename columns to distinguish between g2 and ns3 after merging
g2_subset = g2_subset.rename(columns={
    'node_id': 'g2_node_id',
    'issue_tick': 'g2_issue_tick',
    'callback_tick': 'g2_callback_tick',
    'elapsed_time': 'g2_elapsed_time'
})
ns3_subset = ns3_subset.rename(columns={
    'node_id': 'ns3_node_id',
    'issue_tick': 'ns3_issue_tick',
    'callback_tick': 'ns3_callback_tick',
    'elapsed_time': 'ns3_elapsed_time'
})
analytical_subset = analytical_subset.rename(columns={
    'node_id': 'analytical_node_id',
    'issue_tick': 'analytical_issue_tick',
    'callback_tick': 'analytical_callback_tick',
    'elapsed_time': 'analytical_elapsed_time'
})
# Merge the two dataframes on the GPU ID and the operation name
merged_df = pd.merge(g2_subset, ns3_subset, on=['sys_id', 'node_name'], how='left')
merged_df = pd.merge(merged_df, analytical_subset, on=['sys_id', 'node_name'], how='left')

# --- 3. Visualize Callback Tick Divergence for Each GPU ---

# Sort the data chronologically based on the g2 simulation's start time
merged_df_sorted = merged_df.sort_values(by=['sys_id', 'g2_issue_tick']).reset_index(drop=True)


print("\n--- Plotting Callback Tick Comparison for Each NPU ---")
print("Generating a plot for each NPU to visualize the divergence in operation completion times ('callback_tick').\n")

# Get the list of unique GPUs (sys_id)
gpus = merged_df_sorted['sys_id'].unique()
gpus.sort()

# --- 4. Parse Workload Groups and Assign Markers ---

# Ensure we can import from the root
sys.path.append('/app/astra-sim')
from upc.generate_workloads_split import create_collectives_log

# Define the workload folder (containing the original ET files)
workload_folder = '/app/astra-sim/upc/comparing_networks/workload/GPT_3_1300M_split'
workload_log_path = os.path.join(workload_folder, 'duplicate_collectives.log')

# Generate the log if it doesn't exist
if not os.path.exists(workload_log_path):
    create_collectives_log(workload_folder)

# Define a list of markers to cycle through
markers = ['circle', 'square', 'diamond', 'cross', 'x', 'star', 'hexagram', 'triangle-up', 'triangle-down', 'pentagon']
workload_to_marker = {}
workload_to_signature = {}
collective_to_workload = {}
current_workload = None # Define to avoid UnboundLocalError

try:
    with open(workload_log_path, 'r') as f:
        lines = f.readlines()
        i = 0
        while i < len(lines):
            line = lines[i]
            workload_match = re.match(r'^Workload: (\S+)', line)
            if workload_match:
                current_workload = workload_match.group(1)
                if current_workload not in workload_to_marker:
                    marker_index = len(workload_to_marker) % len(markers)
                    workload_to_marker[current_workload] = markers[marker_index]
                
                if (i + 1 < len(lines)) and (signature_match := re.match(r'^\s+Signature: (.*)', lines[i+1])):
                    workload_to_signature[current_workload] = signature_match.group(1).strip()
            
            collective_match = re.match(r'^\s+-\s(.+)', line)
            if collective_match and current_workload:
                collective_name = collective_match.group(1).strip()
                collective_to_workload[collective_name] = current_workload
            i += 1
    print(f"✅ Successfully parsed {len(workload_to_marker)} workload groups from log file.")
except FileNotFoundError:
    print(f"❌ Warning: Workload log file not found at {workload_log_path}. Using default markers.")
    collective_to_workload = {}

# Create a mapping from node_name to marker symbol
default_marker = 'circle'
merged_df_sorted['marker'] = merged_df_sorted['node_name'].apply(
    lambda name: workload_to_marker.get(collective_to_workload.get(name), default_marker)
)

# --- 5. Generate Plot ---
from plotly.subplots import make_subplots
import math

rows = 4
cols = 4
fig = make_subplots(
    rows=rows, 
    cols=cols, 
    subplot_titles=[f'NPU {gpu_id}' for gpu_id in gpus[:rows*cols]],
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.01,
    horizontal_spacing=0.01
)

for i, gpu_id in enumerate(gpus):
    if i >= rows * cols:
        print(f"Warning: Only plotting first {rows*cols} of {len(gpus)} GPUs.")
        break

    row = (i // cols) + 1
    col = (i % cols) + 1

    gpu_df = merged_df_sorted[merged_df_sorted['sys_id'] == gpu_id].copy()
    operation_order = np.arange(len(gpu_df))
    
    g2_x, g2_y, g2_hover, g2_markers = [], [], [], []
    ns3_x, ns3_y, ns3_hover, ns3_markers = [], [], [], []
    analytical_x, analytical_y, analytical_hover, analytical_markers = [], [], [], []

    for j, op_idx in enumerate(operation_order):
        node_name = gpu_df['node_name'].iloc[j]
        marker_symbol = gpu_df['marker'].iloc[j]
        
        # --- g2 Trace ---
        g2_node_id = gpu_df['g2_node_id'].iloc[j]
        g2_issue = gpu_df['g2_issue_tick'].iloc[j]
        g2_cb = gpu_df['g2_callback_tick'].iloc[j]
        if pd.notna(g2_issue):
            if pd.notna(g2_cb): # Finished op: line with two markers
                g2_x.extend([g2_issue, g2_cb, None])
                g2_y.extend([op_idx, op_idx, None])
                g2_hover.extend([f"ID: {g2_node_id}<br>{node_name}", f"ID: {g2_node_id}<br>{node_name}", None])
                g2_markers.extend([marker_symbol, marker_symbol, marker_symbol])
            else: # Issued but not finished: single point
                g2_x.extend([g2_issue, None])
                g2_y.extend([op_idx, None])
                g2_hover.extend([f"ID: {g2_node_id}<br>{node_name}", None])
                g2_markers.extend([marker_symbol, marker_symbol])

        # --- ns3 Trace ---
        ns3_node_id = gpu_df['ns3_node_id'].iloc[j]
        ns3_issue = gpu_df['ns3_issue_tick'].iloc[j]
        ns3_cb = gpu_df['ns3_callback_tick'].iloc[j]
        if pd.notna(ns3_issue):
            if pd.notna(ns3_cb): # Finished op
                ns3_x.extend([ns3_issue, ns3_cb, None])
                ns3_y.extend([op_idx, op_idx, None])
                ns3_hover.extend([f"ID: {ns3_node_id}<br>{node_name}", f"ID: {ns3_node_id}<br>{node_name}", None])
                ns3_markers.extend([marker_symbol, marker_symbol, marker_symbol])
            else: # Issued but not finished
                ns3_x.extend([ns3_issue, None])
                ns3_y.extend([op_idx, None])
                ns3_hover.extend([f"ID: {ns3_node_id}<br>{node_name}", None])
                ns3_markers.extend([marker_symbol, marker_symbol])

        # --- analytical Trace ---
        analytical_node_id = gpu_df['analytical_node_id'].iloc[j]
        analytical_issue = gpu_df['analytical_issue_tick'].iloc[j]
        analytical_cb = gpu_df['analytical_callback_tick'].iloc[j]
        if pd.notna(analytical_issue):
            if pd.notna(analytical_cb): # Finished op
                analytical_x.extend([analytical_issue, analytical_cb, None])
                analytical_y.extend([op_idx, op_idx, None])
                analytical_hover.extend([f"ID: {analytical_node_id}<br>{node_name}", f"ID: {analytical_node_id}<br>{node_name}", None])
                analytical_markers.extend([marker_symbol, marker_symbol, marker_symbol])
            else: # Issued but not finished
                analytical_x.extend([analytical_issue, None])
                analytical_y.extend([op_idx, None])
                analytical_hover.extend([f"ID: {analytical_node_id}<br>{node_name}", None])
                analytical_markers.extend([marker_symbol, marker_symbol])

    fig.add_trace(go.Scatter(
        x=g2_x, y=g2_y, mode='lines+markers', name='g2', line=dict(color='blue'),
        marker=dict(size=6, symbol=g2_markers), hovertext=g2_hover,
        hovertemplate='<b>%{hovertext}</b><br>Time: %{x}<br>Order: %{y}<extra></extra>',
        legendgroup='g2', showlegend=(i==0)
    ), row=row, col=col)
    
    fig.add_trace(go.Scatter(
        x=ns3_x, y=ns3_y, mode='lines+markers', name='ns3', line=dict(color='orange'),
        marker=dict(size=6, symbol=ns3_markers), hovertext=ns3_hover,
        hovertemplate='<b>%{hovertext}</b><br>Time: %{x}<br>Order: %{y}<extra></extra>',
        legendgroup='ns3', showlegend=(i==0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=analytical_x, y=analytical_y, mode='lines+markers', name='analytical', line=dict(color='green'),
        marker=dict(size=6, symbol=analytical_markers), hovertext=analytical_hover,
        hovertemplate='<b>%{hovertext}</b><br>Time: %{x}<br>Order: %{y}<extra></extra>',
        legendgroup='analytical', showlegend=(i==0)
    ), row=row, col=col)

# for workload_name, marker in workload_to_marker.items():
#     signature = workload_to_signature.get(workload_name, "N/A")
#     fig.add_trace(go.Scatter(
#         x=[None], y=[None], mode='markers',
#         marker=dict(symbol=marker, color='black', size=8),
#         name=f"{workload_name}<br>  └─ Signature: {signature}",
#         legendgroup='Workloads', showlegend=True
#     ), row=1, col=1)

fig.update_layout(
    title_text='Operation Duration (Issue to Callback) for All NPUs',
    height=1000, width=1000, legend_title="Simulation & Workloads",
    # legend=dict(tracegroupgap=20)
)

# Show x-axis titles only on the last row
fig.update_xaxes(title_text='Time (ticks)', row=rows, col=1)
fig.update_xaxes(title_text='Time (ticks)', row=rows, col=2)
fig.update_xaxes(title_text='Time (ticks)', row=rows, col=3)
fig.update_xaxes(title_text='Time (ticks)', row=rows, col=4)

# Show y-axis titles only on the first column
fig.update_yaxes(title_text='Comm. Op. Order', row=1, col=1)
fig.update_yaxes(title_text='Comm. Op. Order', row=2, col=1)
fig.update_yaxes(title_text='Comm. Op. Order', row=3, col=1)
fig.update_yaxes(title_text='Comm. Op. Order', row=4, col=1)


output_filename = "npu_timing_comparison.pdf"
fig.write_image(output_filename)
    
fig.show()

print(f"✅ Generated plots for all {len(gpus)} NPUs.")
print(f"✅ Plot saved to {output_filename}")


✅ Successfully loaded and processed g2, ns3, and analytical data files.
Filtered for communication nodes. Found 880 comm nodes in g2, 880 in ns3, and 880 in analytical.

--- Plotting Callback Tick Comparison for Each NPU ---
Generating a plot for each NPU to visualize the divergence in operation completion times ('callback_tick').

✅ Successfully parsed 110 workload groups from log file.


✅ Generated plots for all 16 NPUs.
✅ Plot saved to npu_timing_comparison.pdf


In [ ]:
merged_df_sorted[(merged_df_sorted['sys_id'] == 14)].sort_values(by='g2_issue_tick')

,sys_id,node_name,g2_issue_tick,g2_callback_tick,g2_elapsed_time,ns3_issue_tick,ns3_callback_tick,ns3_elapsed_time
1932,14,shadow_mb0.transformer.11.ffn_res.y@0_Y_RECV,0,34480954160618,34480954160618,0,58896235505536,58896235505536
1933,14,mb0.transformer.12.mha.qkv@0_X1COMM,34480954320877,34676080959464,195126638587,58896235665795,59726524045181,830288379386
1934,14,mb0.transformer.12.mha.dwqkv@0_X2_COMM,34676080959464,35195939299734,519858340270,59726524045181,60006848402765,280324357584
1935,14,mb0.transformer.12.mha.o@0_X1COMM,35195939299734,35391066419099,195127119365,60006848402765,60504167789327,497319386562
1936,14,mb0.transformer.12.ffn.x00@0_X1COMM,35391066739617,35910924278596,519857538979,60504168109845,61193544877903,689376768058
...,...,...,...,...,...,...,...,...
2065,14,mb0.transformer.12.ffn.dx0@0_X1COMM,185629939083809,185629957352478,18268669,154176444474380,154176444955177,480797
2066,14,mb0.transformer.12.mha.do1@0_X1COMM,185629957672996,187051904812455,1421947139459,154176445275695,155087148296903,910703021208
2067,14,mb0.transformer.12._sharded_grad@0_X1COMM,187051911653232,188441217889856,1389306236624,155087155137680,155666798360773,579643223093
2068,14,mb0.transformer.12.mha.dx@0_X1COMM,188441217889856,188441218370653,480797,155666798360773,156462643468963,795845108190
